## Word count problem using stream processing !

In [0]:
project_dir = "dbfs:/FileStore/project2/"

In [0]:

from pyspark.sql.functions import trim, split, lower, explode

class StreamProcessingWordCount:
    def __init__(self, project_dir:str):
        self.project_dir=project_dir
        self.landing_zone="landing_zone/"
        self.dataset_dir="dataset/"
        self.checkpoint_dir="checkpoint/"
        self.load_table_name="proj2_word_count_table"
        self.result_df=None


    def cleanup_n_setup(self):
        # drop load table
        drop_query=f"drop table if exists {self.load_table_name}"
        spark.sql(drop_query)

        # drop the load table data storage dirs
        dbutils.fs.rm(f"/user/hive/warehouse/{self.load_table_name}", True)

        # drop the checkpoint and landing dirs
        dbutils.fs.rm(self.project_dir+self.checkpoint_dir, True)
        dbutils.fs.rm(self.project_dir+self.landing_zone, True)

        # Creat the checkPoint and landing zone
        dbutils.fs.mkdirs(self.project_dir+self.checkpoint_dir)
        dbutils.fs.mkdirs(self.project_dir+self.landing_zone)

        print("CLEANUP & SETUP RES: cleanup and setup completed successfully !")


    def ingest_data(self, fileName:str):
        dbutils.fs.cp(self.project_dir+self.dataset_dir+fileName, self.project_dir+self.landing_zone)
        print("INGESTION RES: text file have been ingested to the landing zone successfully !")


    def extract_data(self):
        df=spark.readStream.format("text").option("lineSep",".").load(self.project_dir+self.landing_zone+"*.txt")
        print("EXTRACTION RES: data extraction from landing zone successfull !")
        return df
    
    def transform_data(self, ingested_df):
        word_df=ingested_df.select(explode(split(ingested_df.value," ")).alias("word"))
        quality_df=word_df.select(trim(lower(word_df.word)).alias("word"))
        transformed_df=quality_df.where("word is not null").where("word rlike '[a-z]'")
        transformed_df=transformed_df.groupBy("word").count()
        self.result_df=transformed_df
        print("TRANSFORMATION RES: data have been transformed successfully !")
        return transformed_df

    def load_transformed_data(self, tranformed_df):
        write_stream_query=tranformed_df.writeStream.format("delta").option("checkpointLocation",self.project_dir+self.checkpoint_dir).outputMode("complete").toTable(self.load_table_name)
        print("LOAD RES: data have been loaded successfully !")
        return write_stream_query

    def process_data(self, file_name:str):
        self.cleanup_n_setup()
        self.ingest_data(fileName=file_name)
        extracted_data_df=self.extract_data()
        transformed_df=self.transform_data(extracted_data_df)
        streamingQuery=self.load_transformed_data(transformed_df)
        print("PROCESSING RESULT: data processing have been completed !\n STRAMING QUERY STARTED SUCCESSFULLY !")
        return streamingQuery

    



### `project_2_word_count_stream_processing` script run successfull !